In [1]:
pip install billboard.py pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import billboard


# =========================
# 1. 수집할 빌보드 차트 설정
# =========================

CHARTS = {
    "hot-100": "Pop / All Genre",
    "r-b-hip-hop-songs": "R&B / Hip-Hop",
    "rap-song": "Rap",
    "country-songs": "Country",
    "rock-songs": "Rock / Alternative",
    "dance-electronic-songs": "Dance / Electronic",
    "latin-songs": "Latin"
}

# None이면 최신 차트를 가져옵니다.
# 특정 날짜를 넣고 싶으면 예: "2026-07-04"
TARGET_DATE = None

# 차트별 최대 수집 개수
MAX_ROWS_PER_CHART = 100

# 저장 파일명
OUTPUT_CSV = "billboard_chart_dataset.csv"
OUTPUT_XLSX = "billboard_chart_dataset.xlsx"


# =========================
# 2. 텍스트 정리 함수
# =========================

def clean_text(text):
    if text is None:
        return ""

    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_text(text):
    text = clean_text(text).lower()
    text = re.sub(r"[^a-z0-9가-힣 ]", "", text)
    return text.strip()


# =========================
# 3. 차트 진입일 추정 함수
# =========================

def estimate_entry_date(chart_date, weeks_on_chart):
    """
    Billboard 현재 차트에는 weeks on chart가 제공됩니다.
    이를 이용해 대략적인 차트 진입일을 계산합니다.

    주의:
    곡이 차트에서 빠졌다가 다시 진입한 경우,
    실제 최초 진입일과 다를 수 있습니다.
    """

    if not chart_date or not weeks_on_chart:
        return ""

    try:
        base_date = datetime.strptime(chart_date, "%Y-%m-%d")
        weeks = int(weeks_on_chart)

        estimated_date = base_date - timedelta(days=7 * (weeks - 1))
        return estimated_date.strftime("%Y-%m-%d")

    except Exception:
        return ""


# =========================
# 4. 곡 분위기 자동 분류 함수
# =========================

def classify_mood(title, genre):
    title_norm = normalize_text(title)

    mood_rules = [
        {
            "keywords": ["love", "heart", "kiss", "sweet", "baby", "romance"],
            "mood": "로맨틱"
        },
        {
            "keywords": ["sad", "cry", "tears", "lonely", "alone", "hurt", "miss"],
            "mood": "슬픔 / 감성"
        },
        {
            "keywords": ["dance", "party", "club", "night", "shake", "jump"],
            "mood": "신남 / 댄스"
        },
        {
            "keywords": ["bad", "fight", "war", "rage", "kill", "monster"],
            "mood": "강렬함 / 반항"
        },
        {
            "keywords": ["dream", "moon", "blue", "rain", "slow", "sleep"],
            "mood": "몽환 / 차분"
        }
    ]

    for rule in mood_rules:
        for keyword in rule["keywords"]:
            if keyword in title_norm:
                return rule["mood"]

    genre_mood_map = {
        "Pop / All Genre": "대중적 / 트렌디",
        "R&B / Hip-Hop": "그루브 / 세련됨",
        "Rap": "강렬함 / 자기표현",
        "Country": "따뜻함 / 서정적",
        "Rock / Alternative": "개성적 / 강렬함",
        "Dance / Electronic": "신남 / 에너지",
        "Latin": "열정적 / 리듬감"
    }

    return genre_mood_map.get(genre, "일반")


# =========================
# 5. 이미지 키워드 생성 함수
# =========================

def make_image_keywords(genre, mood):
    genre_keywords = {
        "Pop / All Genre": ["트렌디", "도시적", "대중적", "밝은 색감"],
        "R&B / Hip-Hop": ["스트리트", "고급스러움", "그루브", "시크"],
        "Rap": ["강렬함", "스트리트 패션", "자기표현", "힙합 무드"],
        "Country": ["자연", "따뜻함", "빈티지", "편안함"],
        "Rock / Alternative": ["스모키", "개성", "다크톤", "반항적"],
        "Dance / Electronic": ["네온", "클럽", "화려함", "에너지"],
        "Latin": ["열정", "강렬한 색감", "리듬", "글램"]
    }

    mood_keywords = {
        "로맨틱": ["핑크", "부드러운 조명", "데이트", "러블리"],
        "슬픔 / 감성": ["블루톤", "감성적", "비 오는 거리", "차분함"],
        "신남 / 댄스": ["네온", "파티", "글리터", "화려함"],
        "강렬함 / 반항": ["레드", "스모키", "강한 대비", "카리스마"],
        "몽환 / 차분": ["파스텔", "몽환적", "달빛", "부드러움"],
        "대중적 / 트렌디": ["트렌디", "깔끔함", "도시", "데일리"],
        "그루브 / 세련됨": ["세련됨", "무드 조명", "고급스러움", "감각적"],
        "강렬함 / 자기표현": ["강렬함", "스트리트", "힙합", "자신감"],
        "따뜻함 / 서정적": ["자연", "따뜻함", "노을", "편안함"],
        "개성적 / 강렬함": ["다크톤", "개성", "스모키", "록스타일"],
        "신남 / 에너지": ["네온", "에너지", "클럽", "댄스"],
        "열정적 / 리듬감": ["레드", "라틴", "열정", "리듬"]
    }

    keywords = []

    keywords.extend(genre_keywords.get(genre, []))
    keywords.extend(mood_keywords.get(mood, []))

    unique_keywords = []
    for keyword in keywords:
        if keyword not in unique_keywords:
            unique_keywords.append(keyword)

    return ", ".join(unique_keywords)


# =========================
# 6. 빌보드 차트 수집 함수
# =========================

def collect_billboard_chart(chart_name, genre, target_date=None):
    rows = []

    try:
        if target_date:
            chart = billboard.ChartData(chart_name, date=target_date)
        else:
            chart = billboard.ChartData(chart_name)

        chart_date = chart.date

        print(f"[수집 중] {chart_name} / {genre} / 차트 날짜: {chart_date}")

        for entry in chart[:MAX_ROWS_PER_CHART]:
            song_title = clean_text(entry.title)
            artist_name = clean_text(entry.artist)

            rank = entry.rank
            peak_rank = getattr(entry, "peakPos", "")
            weeks_on_chart = getattr(entry, "weeks", "")

            entry_date = estimate_entry_date(chart_date, weeks_on_chart)

            mood = classify_mood(song_title, genre)
            image_keywords = make_image_keywords(genre, mood)

            rows.append({
                "가수명": artist_name,
                "곡명": song_title,
                "장르": genre,
                "순위": rank,
                "차트날짜": chart_date,
                "차트진입일_추정": entry_date,
                "최고순위": peak_rank,
                "차트유지기간_주": weeks_on_chart,
                "곡분위기": mood,
                "이미지키워드": image_keywords
            })

    except Exception as e:
        print(f"[오류] {chart_name} 수집 실패: {e}")

    return rows


# =========================
# 7. 전체 수집 실행
# =========================

def main():
    all_rows = []

    for chart_name, genre in CHARTS.items():
        chart_rows = collect_billboard_chart(
            chart_name=chart_name,
            genre=genre,
            target_date=TARGET_DATE
        )

        all_rows.extend(chart_rows)

    if not all_rows:
        print("수집된 데이터가 없습니다.")
        return

    df = pd.DataFrame(all_rows)

    # 중복 제거
    df = df.drop_duplicates(subset=["가수명", "곡명", "장르", "차트날짜"])

    # 순위 기준 정렬
    df = df.sort_values(by=["장르", "순위"])

    # 저장
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    df.to_excel(OUTPUT_XLSX, index=False)

    print("\n수집 완료")
    print(f"CSV 저장: {OUTPUT_CSV}")
    print(f"Excel 저장: {OUTPUT_XLSX}")
    print("\n미리보기")
    print(df.head(20))


if __name__ == "__main__":
    main()

[오류] hot-100 수집 실패: module 'billboard' has no attribute 'ChartData'
[오류] r-b-hip-hop-songs 수집 실패: module 'billboard' has no attribute 'ChartData'
[오류] rap-song 수집 실패: module 'billboard' has no attribute 'ChartData'
[오류] country-songs 수집 실패: module 'billboard' has no attribute 'ChartData'
[오류] rock-songs 수집 실패: module 'billboard' has no attribute 'ChartData'
[오류] dance-electronic-songs 수집 실패: module 'billboard' has no attribute 'ChartData'
[오류] latin-songs 수집 실패: module 'billboard' has no attribute 'ChartData'
수집된 데이터가 없습니다.


In [4]:
# music_beauty_recommender.py

def get_recommendation_data():
    return {
        "Pop": {
            "추천화장품유형": ["쿠션", "틴트", "글로우 립", "데일리 아이섀도"],
            "추천메이크업스타일": "트렌디하고 깔끔한 데일리 메이크업",
            "추천향수계열": "플로럴, 시트러스",
            "브랜드이미지": "밝고 대중적이며 세련된 이미지",
            "추천컬러": ["코랄", "핑크", "피치", "로즈"],
            "추천이유": "Pop은 대중적이고 밝은 이미지가 강해서 자연스럽고 트렌디한 화장품과 잘 어울립니다."
        },
        "R&B / Hip-Hop": {
            "추천화장품유형": ["립스틱", "아이섀도", "아이라이너", "하이라이터"],
            "추천메이크업스타일": "세련되고 시크한 스트리트 메이크업",
            "추천향수계열": "머스크, 우디, 앰버",
            "브랜드이미지": "도시적이고 고급스러운 이미지",
            "추천컬러": ["브라운", "누드", "버건디", "딥로즈"],
            "추천이유": "R&B와 Hip-Hop은 감각적이고 개성적인 분위기가 강해서 립, 섀도우, 시크한 메이크업과 잘 맞습니다."
        },
        "Rap": {
            "추천화장품유형": ["매트 립", "쉐딩", "아이라이너", "강한 색조 제품"],
            "추천메이크업스타일": "강렬하고 자기표현이 뚜렷한 메이크업",
            "추천향수계열": "스파이시, 우디, 레더",
            "브랜드이미지": "강렬하고 자신감 있는 이미지",
            "추천컬러": ["레드", "블랙", "딥브라운", "와인"],
            "추천이유": "Rap은 강한 자기표현과 개성이 중요하므로 선명한 색조와 카리스마 있는 스타일이 어울립니다."
        },
        "Dance / Electronic": {
            "추천화장품유형": ["글리터", "하이라이터", "향수", "컬러 아이섀도"],
            "추천메이크업스타일": "화려하고 에너지 있는 네온 메이크업",
            "추천향수계열": "프루티, 시트러스, 아쿠아",
            "브랜드이미지": "화려하고 젊고 에너지 넘치는 이미지",
            "추천컬러": ["실버", "퍼플", "블루", "핫핑크"],
            "추천이유": "Dance와 Electronic은 클럽, 네온, 에너지 이미지가 강해서 글리터와 화려한 색조가 잘 어울립니다."
        },
        "Country": {
            "추천화장품유형": ["스킨케어", "톤업크림", "누드 립", "내추럴 블러셔"],
            "추천메이크업스타일": "편안하고 자연스러운 내추럴 메이크업",
            "추천향수계열": "우디, 허브, 그린",
            "브랜드이미지": "자연스럽고 따뜻하며 편안한 이미지",
            "추천컬러": ["베이지", "브라운", "살구", "누드"],
            "추천이유": "Country는 자연, 따뜻함, 편안함의 이미지가 강해서 과하지 않은 내추럴 제품과 잘 맞습니다."
        },
        "Latin": {
            "추천화장품유형": ["레드 립", "글로시 립", "블러셔", "향수"],
            "추천메이크업스타일": "열정적이고 글램한 메이크업",
            "추천향수계열": "플로럴, 프루티, 바닐라",
            "브랜드이미지": "강렬하고 화려하며 매혹적인 이미지",
            "추천컬러": ["레드", "오렌지", "골드", "코랄"],
            "추천이유": "Latin은 리듬감과 열정적인 분위기가 강해서 레드립, 글램 메이크업, 향수와 잘 어울립니다."
        },
        "Rock / Alternative": {
            "추천화장품유형": ["스모키 아이섀도", "아이라이너", "다크 립", "매트 파운데이션"],
            "추천메이크업스타일": "개성 있고 강렬한 스모키 메이크업",
            "추천향수계열": "우디, 머스크, 스모키",
            "브랜드이미지": "개성적이고 반항적이며 독립적인 이미지",
            "추천컬러": ["블랙", "그레이", "버건디", "딥레드"],
            "추천이유": "Rock과 Alternative는 개성과 반항적인 이미지가 강해서 스모키 메이크업과 다크톤 색조가 어울립니다."
        },
        "K-Pop": {
            "추천화장품유형": ["쿠션", "틴트", "글리터 섀도우", "블러셔"],
            "추천메이크업스타일": "화사하고 깨끗한 아이돌 메이크업",
            "추천향수계열": "플로럴, 프루티, 파우더리",
            "브랜드이미지": "귀엽고 트렌디하며 화사한 이미지",
            "추천컬러": ["핑크", "코랄", "라벤더", "피치"],
            "추천이유": "K-Pop은 화사하고 세련된 이미지가 강해서 깨끗한 피부 표현과 틴트, 글리터 제품이 잘 어울립니다."
        }
    }


def show_genres(data):
    print("\n선택 가능한 음악 장르")
    print("-" * 30)

    genres = list(data.keys())

    for i, genre in enumerate(genres, start=1):
        print(f"{i}. {genre}")

    return genres


def recommend_by_genre(selected_genre, data):
    if selected_genre not in data:
        return None

    return data[selected_genre]


def print_recommendation(genre, result):
    print("\n" + "=" * 50)
    print(f"선택한 음악 장르: {genre}")
    print("=" * 50)

    print("\n추천 화장품 유형")
    for item in result["추천화장품유형"]:
        print(f"- {item}")

    print("\n추천 메이크업 스타일")
    print(result["추천메이크업스타일"])

    print("\n추천 향수 계열")
    print(result["추천향수계열"])

    print("\n어울리는 브랜드 이미지")
    print(result["브랜드이미지"])

    print("\n추천 컬러")
    for color in result["추천컬러"]:
        print(f"- {color}")

    print("\n추천 이유")
    print(result["추천이유"])

    print("\n" + "=" * 50)


def main():
    data = get_recommendation_data()
    genres = show_genres(data)

    try:
        choice = int(input("\n좋아하는 음악 장르 번호를 입력하세요: "))

        if choice < 1 or choice > len(genres):
            print("잘못된 번호입니다.")
            return

        selected_genre = genres[choice - 1]
        result = recommend_by_genre(selected_genre, data)

        print_recommendation(selected_genre, result)

    except ValueError:
        print("숫자로 입력해 주세요.")


if __name__ == "__main__":
    main()


선택 가능한 음악 장르
------------------------------
1. Pop
2. R&B / Hip-Hop
3. Rap
4. Dance / Electronic
5. Country
6. Latin
7. Rock / Alternative
8. K-Pop

선택한 음악 장르: K-Pop

추천 화장품 유형
- 쿠션
- 틴트
- 글리터 섀도우
- 블러셔

추천 메이크업 스타일
화사하고 깨끗한 아이돌 메이크업

추천 향수 계열
플로럴, 프루티, 파우더리

어울리는 브랜드 이미지
귀엽고 트렌디하며 화사한 이미지

추천 컬러
- 핑크
- 코랄
- 라벤더
- 피치

추천 이유
K-Pop은 화사하고 세련된 이미지가 강해서 깨끗한 피부 표현과 틴트, 글리터 제품이 잘 어울립니다.

